# 🚀 BƯỚC 1: Cài đặt Thư viện và Kết nối Google Drive

In [ ]:
!apt-get update
!apt-get install -y tesseract-ocr tesseract-ocr-vie
# Bổ sung thư viện siêu AI bóc tách PDF: marker-pdf và easyocr
!pip install google-generativeai supabase pypdf pymupdf pytesseract marker-pdf easyocr
from google.colab import drive
drive.mount('/content/drive')
print('✅ Đã cài đặt và kết nối thành công!')

# ⚙️ BƯỚC 2: Cấu hình API Keys

In [ ]:
import google.generativeai as genai
from supabase import create_client

# BẠN HÃY DÁN CÁC KEY CỦA BẠN VÀO DANH SÁCH NÀY NHÉ:
GEMINI_KEYS = [
    "ĐIỀN_KEY_1_VÀO_ĐÂY",
    "ĐIỀN_KEY_2_VÀO_ĐÂY",
    "ĐIỀN_KEY_3_VÀO_ĐÂY",
    "ĐIỀN_KEY_4_VÀO_ĐÂY",
    "ĐIỀN_KEY_5_VÀO_ĐÂY",
    "ĐIỀN_KEY_6_VÀO_ĐÂY",
]

GEMINI_KEYS = [k.strip() for k in GEMINI_KEYS if k.strip() and "ĐIỀN_KEY" not in k]
current_key_index = 0
if GEMINI_KEYS:
    genai.configure(api_key=GEMINI_KEYS[current_key_index])

SUPABASE_URL = "ĐIỀN_SUPABASE_URL_VÀO_ĐÂY".strip()
SUPABASE_KEY = "ĐIỀN_SUPABASE_KEY_VÀO_ĐÂY".strip()
supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

print('✅ Đã cấu hình API thành công!')

# 📁 BƯỚC 3: Cấu hình Thư mục Sách giáo khoa

In [ ]:
import os
BASE_DIR = '/content/drive/MyDrive/SGK_Moi_NXBGĐ'

if not os.path.exists(BASE_DIR):
    print(f'❌ Không tìm thấy thư mục {BASE_DIR}')
else:
    print(f'✅ Đã tìm thấy thư mục SGK tại: {BASE_DIR}')

# 🤖 BƯỚC 4: Kích hoạt Nhà máy Xử lý Dữ liệu (BẢN FLASH - BỎ QUA GOOGLE)

In [ ]:
import time
import uuid
import fitz  # PyMuPDF
from io import BytesIO
from PIL import Image
import pytesseract
import concurrent.futures
import unicodedata
import shutil
import subprocess
import gc
import torch
from google.generativeai.types import HarmCategory, HarmBlockThreshold

MARKER_MODELS = None
MARKER_CONVERTER = None
MARKER_PYTHON_API_AVAILABLE = True
EASYOCR_READER = None

def rotate_gemini_key():
    global current_key_index
    current_key_index = (current_key_index + 1) % len(GEMINI_KEYS)
    new_key = GEMINI_KEYS[current_key_index]
    genai.configure(api_key=new_key)
    print(f"\n🔄 [HỆ THỐNG] Đã tự động chuyển sang dùng API Key số {current_key_index + 1}")

def upload_bytes_to_storage(file_bytes, storage_path, content_type):
    supabase.storage.from_("giasuao").upload(storage_path, file_bytes, file_options={"content-type": content_type, "upsert": "true"})
    return supabase.storage.from_("giasuao").get_public_url(storage_path)

def run_marker_python_api(pdf_path, filename):
    global MARKER_MODELS, MARKER_CONVERTER
    if MARKER_CONVERTER is None:
        print(f"\n⚡ Đang khởi động Siêu AI Marker-PDF (CHẾ ĐỘ CHUYÊN TOÁN)... ")
        try:
            from marker.converters.pdf import PdfConverter
            from marker.models import create_model_dict
            MARKER_MODELS = create_model_dict()
            MARKER_CONVERTER = PdfConverter(artifact_dict=MARKER_MODELS)
        except ImportError:
            from marker.models import load_all_models
            MARKER_MODELS = load_all_models()
            MARKER_CONVERTER = "v0.2"

    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()
    
    CHUNK_SIZE = 30
    full_markdown = ""
    
    for start_page in range(0, total_pages, CHUNK_SIZE):
        end_page = min(start_page + CHUNK_SIZE, total_pages)
        print(f"\n   ✂️ Đang nạp khúc Toán {start_page+1} đến {end_page}... ", end="")
        
        chunk_pdf_path = f"/tmp/chunk_{start_page}.pdf"
        chunk_doc = fitz.open()
        chunk_doc.insert_pdf(fitz.open(pdf_path), from_page=start_page, to_page=end_page-1)
        chunk_doc.save(chunk_pdf_path)
        chunk_doc.close()
        
        print("Đang giải mã Công thức: ", end="", flush=True)
        
        if MARKER_CONVERTER == "v0.2":
            from marker.convert import convert_single_pdf
            text, _, _ = convert_single_pdf(chunk_pdf_path, MARKER_MODELS)
            full_markdown += text + "\n\n"
        else:
            from marker.output import text_from_rendered
            rendered = MARKER_CONVERTER(chunk_pdf_path)
            text, _, _ = text_from_rendered(rendered)
            full_markdown += text + "\n\n"
            
        print(" [Hoàn tất khúc Toán!]")
        
        if os.path.exists(chunk_pdf_path): 
            os.remove(chunk_pdf_path)
        
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    if full_markdown:
        print("\n\n [THÀNH CÔNG RỰC RỠ! ĐÃ DỊCH XONG TOÀN BỘ CÔNG THỨC MÔN TỰ NHIÊN!]")
    return full_markdown

def run_marker_subprocess(pdf_path, filename):
    print(f"\n⚡ Đang chạy Marker-PDF bằng chế độ Cơ Bản...")
    out_dir = "/content/marker_out"
    os.makedirs(out_dir, exist_ok=True)
    
    cmd_name = "marker_single" if shutil.which("marker_single") else "marker"
    if not shutil.which(cmd_name):
        return ""

    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()
    
    CHUNK_SIZE = 30
    full_markdown = ""
    
    for start_page in range(0, total_pages, CHUNK_SIZE):
        end_page = min(start_page + CHUNK_SIZE, total_pages)
        chunk_pdf_path = f"/tmp/chunk_{start_page}.pdf"
        chunk_doc = fitz.open()
        chunk_doc.insert_pdf(fitz.open(pdf_path), from_page=start_page, to_page=end_page-1)
        chunk_doc.save(chunk_pdf_path)
        chunk_doc.close()
        
        chunk_out_dir = f"{out_dir}/chunk_{start_page}"
        os.makedirs(chunk_out_dir, exist_ok=True)
        cmd = [cmd_name, chunk_pdf_path, "--output_dir", chunk_out_dir] if cmd_name == "marker_single" else [cmd_name, chunk_pdf_path, chunk_out_dir]
            
        try:
            process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            process.wait()
            if process.returncode == 0 or process.returncode is None:
                base_name = os.path.splitext(os.path.basename(chunk_pdf_path))[0]
                md_file = os.path.join(chunk_out_dir, base_name, f"{base_name}.md")
                if not os.path.exists(md_file):
                    md_file = os.path.join(chunk_out_dir, f"{base_name}.md")
                if os.path.exists(md_file):
                    with open(md_file, "r", encoding="utf-8") as f:
                        full_markdown += f.read() + "\n\n"
        except Exception:
            pass
            
        if os.path.exists(chunk_pdf_path): os.remove(chunk_pdf_path)
        shutil.rmtree(chunk_out_dir, ignore_errors=True)
        gc.collect()
        
    return full_markdown

def run_marker_pdf(pdf_path, filename):
    global MARKER_PYTHON_API_AVAILABLE
    if MARKER_PYTHON_API_AVAILABLE:
        try:
            return run_marker_python_api(pdf_path, filename)
        except Exception as e:
            MARKER_PYTHON_API_AVAILABLE = False
            if torch.cuda.is_available(): torch.cuda.empty_cache()
            gc.collect()
            return run_marker_subprocess(pdf_path, filename)
    return run_marker_subprocess(pdf_path, filename)

def extract_text_from_pdf(pdf_path, filename, grade):
    nfkd_form = unicodedata.normalize('NFKD', filename.lower())
    name_clean = "".join([c for c in nfkd_form if not unicodedata.combining(c)]).replace('đ', 'd')
    tokens = name_clean.split()
    
    math_subjects = ['toan', 'vat li', 'vat ly', 'hoa hoc', 'khoa hoc', 'cong nghe', 'tin hoc']
    needs_latex = any(sub in name_clean for sub in math_subjects)
    
    if 'li' in tokens and 'dia' not in tokens:
        needs_latex = True
    if 'ly' in tokens and 'dia' not in tokens and 'dia ly' not in name_clean:
        needs_latex = True
    if 'hoa' in tokens and 'hoat dong' not in name_clean:
        needs_latex = True
        
    exclude_subjects = ['lich su', 'dia li', 'dia ly', 'tu nhien va xa hoi', 'hoat dong', 'am nhac', 'my thuat', 'mi thuat', 'ngu van', 'tiếng']
    if any(ex in name_clean for ex in exclude_subjects):
        needs_latex = False
        
    # 1. ĐỐI VỚI MÔN TỰ NHIÊN (CẦN CÔNG THỨC LATEX) -> VẪN GIỮ NGUYÊN MARKER NHƯ CŨ!
    if needs_latex:
        print(f"\n🧠 Nhận diện sách '{filename}' thuộc môn Toán/Khoa học (Cần quét công thức).")
        return run_marker_pdf(pdf_path, filename)

    # 2. ĐỐI VỚI MÔN XÃ HỘI -> CẮT BỎ GOOGLE VISION, VẮT KIỆT SỨC MẠNH EASYOCR!
    print(f"\n📚 Nhận diện sách '{filename}' thuộc môn Xã hội (Nhiều chữ).")
    print(f'👉 ĐÃ TẮT GOOGLE VISION! Kích hoạt chế độ EasyOCR (GPU) Tốc độ ánh sáng...')
    
    doc = fitz.open(pdf_path)
    total_pages = len(doc)
    doc.close()
    
    CHUNK_SIZE = 15
    full_book_text = ""
    
    try:
        import easyocr
        import numpy as np
        global EASYOCR_READER
        if 'EASYOCR_READER' not in globals() or EASYOCR_READER is None:
            print(" (Đang khởi động nòng súng EasyOCR lần đầu...)", end="", flush=True)
            EASYOCR_READER = easyocr.Reader(['vi'], gpu=True, verbose=False)
    except Exception as e:
        print(f"\n⚠️ Lỗi khởi tạo EasyOCR ({e}). Chuyển sang Marker...")
        return run_marker_pdf(pdf_path, filename)

    safety_settings = {
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    }
    model_flash = genai.GenerativeModel(model_name="gemini-1.5-flash", safety_settings=safety_settings)
    
    for start_page in range(0, total_pages, CHUNK_SIZE):
        end_page = min(start_page + CHUNK_SIZE, total_pages)
        print(f"\nĐang quét đoạn từ trang {start_page+1} đến {end_page}: ", end="", flush=True)
        
        doc = fitz.open(pdf_path)
        images = []
        for i in range(start_page, end_page):
            page = doc.load_page(i)
            pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            images.append(img)
        doc.close()
        
        chunk_raw_text = ""
        for img in images:
            img_np = np.array(img)
            results = EASYOCR_READER.readtext(img_np, detail=0, paragraph=True)
            chunk_raw_text += "\n".join(results) + "\n\n"
            print("⚡", end="", flush=True)
            
        print(" [Hoàn thành! Đang dàn trang Markdown...]", end="", flush=True)
        
        # Nhờ Gemini dàn trang Text (Tránh lỗi 404 bằng try/except kỹ lưỡng)
        format_prompt = f"Sắp xếp lại văn bản thô sau đây thành định dạng Markdown rõ ràng. Giữ nguyên toàn bộ nội dung:\n\n{chunk_raw_text[:30000]}"
        chunk_text = chunk_raw_text
        
        try:
            res = model_flash.generate_content(format_prompt)
            if res.text:
                chunk_text = res.text
                print(" [Bản đẹp 100%]")
        except Exception:
            print(" [AI bị ngợp, giữ nguyên bản thô]")
            
        full_book_text += chunk_text + "\n\n"
        images.clear()
        gc.collect()
        
    print("\n✅ Đã quét siêu tốc xong 100% cuốn sách Xã hội!")
    return full_book_text

def create_embedding(text):
    safe_text = text[:9000]
    for attempt in range(6):
        try:
            result = genai.embed_content(model="models/gemini-embedding-001", content=safe_text)
            return result['embedding'][:768]
        except Exception as e:
            error_msg = str(e).lower()
            if "429" in error_msg or "quota" in error_msg:
                rotate_gemini_key()
                time.sleep(2)
            else:
                return None
    return None

def process_all():
    books_to_process = []
    for grade in range(1, 13):
        grade_folder = os.path.join(BASE_DIR, f'Lop_{grade}')
        if not os.path.exists(grade_folder):
            continue
        for filename in os.listdir(grade_folder):
            if filename.lower().endswith('.pdf'):
                books_to_process.append((grade, filename, os.path.join(grade_folder, filename)))
                
    print(f"\n🔍 TỔNG KIỂM TRA: Tìm thấy {len(books_to_process)} cuốn sách.")
    print("===============================================================")
    
    for grade, filename, pdf_path in books_to_process:
        res = supabase.table('documents').select('id').eq('name', filename).execute()
        if res.data:
            print(f'\n⏭️ BỎ QUA [{filename}]: Đã có trong Database.')
            continue
            
        print(f'\n🚀 Bắt đầu xử lý sách mới: {filename}')
        doc_id = str(uuid.uuid5(uuid.NAMESPACE_URL, filename))
        
        thumb_url = ""
        try:
            doc = fitz.open(pdf_path)
            page = doc.load_page(0)
            pix = page.get_pixmap(matrix=fitz.Matrix(0.5, 0.5))
            thumb_bytes = pix.tobytes("png")
            doc.close()
            thumb_url = upload_bytes_to_storage(thumb_bytes, f"thumbnails/{doc_id}.png", "image/png")
        except Exception:
            pass
            
        text_content = extract_text_from_pdf(pdf_path, filename, grade)
        if not text_content or len(text_content) < 50:
            print(f"⚠️ THẤT BẠI [{filename}]: Không lấy được chữ.\n")
            continue
            
        vector = create_embedding(text_content)
        if not vector:
            print(f"⚠️ THẤT BẠI [{filename}]: Lỗi Vector.\n")
            continue
            
        doc_data = {
            "id": doc_id,
            "name": filename,
            "pdf_url": "",
            "thumbnail_url": thumb_url,
            "grade": str(grade),
            "content": text_content,
            "embedding": vector,
            "status": "ready"
        }
        supabase.table("documents").insert(doc_data).execute()
        print(f"✅ HOÀN THÀNH [{filename}]: Đã nạp thành công 100% vào Database!\n")
        
    print("\n🎉 TẤT CẢ SÁCH ĐÃ ĐƯỢC XỬ LÝ XONG!")

if os.path.exists(BASE_DIR):
    process_all()
else:
    print("Vui lòng thiết lập chuẩn thư mục Google Drive ở Bước 3 trước nhé!")